In [4]:
# -*- coding: utf-8 -*-

import os

# Nomes dos meses em português (índice 1..12)
MESES_PT = [
    "", "janeiro", "fevereiro", "março", "abril", "maio", "junho",
    "julho", "agosto", "setembro", "outubro", "novembro", "dezembro"
]

# ---------- Leitura / Tratamento dos dados ----------
def carregar_dados(nome_arquivo):

    dados = []
    ignoradas = 0
    if not os.path.exists(nome_arquivo):
        print("Arquivo não encontrado:", nome_arquivo)
        return dados

    with open("Anexo_Arquivo_Dados_Projeto_Logica_e_programacao_de_computadores.csv", "r", encoding="latin1") as arq:
        cab = arq.readline()  # cabeçalho (ignoramos os nomes dos campos do arquivo)
        for linha in arq:
            linha = linha.strip()
            if not linha:
                continue
            partes = linha.split(",")
            # Esperamos pelo menos 8 colunas: data, precip, maxima, minima, horas_insol, temp_media, um_relativa, vel_vento
            if len(partes) < 8:
                ignoradas += 1
                continue
            try:
                data_str = partes[0].strip()            # dd/mm/aaaa
                d_str, m_str, y_str = data_str.split("/")
                dia = int(d_str)
                mes = int(m_str)
                ano = int(y_str)
                precip = float(partes[1])
                maxima = float(partes[2])
                minima = float(partes[3])
                # partes[4] = horas_insol (usamos se precisar)
                # partes[5] = temp_media (não usado diretamente nos itens ped)
                umidade = float(partes[6])
                vento = float(partes[7])
            except Exception:
                ignoradas += 1
                continue

            reg = {
                "data": data_str,
                "dia": dia,
                "mes": mes,
                "ano": ano,
                "precip": precip,
                "maxima": maxima,
                "minima": minima,
                "umidade": umidade,
                "vento": vento
            }
            dados.append(reg)
    if ignoradas > 0:
        print(f"Atenção: {ignoradas} linha(s) foram ignoradas por formato inválido.")
    return dados

def anos_disponiveis(dados):
    anos = set()
    for r in dados:
        anos.add(r["ano"])
    if not anos:
        return None, None
    return min(anos), max(anos)

# A) Visualização de intervalo
def visualizar_intervalo(dados):
    if not dados:
        print("Nenhum dado carregado.")
        return

    ano_min, ano_max = anos_disponiveis(dados)
    print(f"Dados disponíveis entre {ano_min} e {ano_max}.\n")

    #Validação do período
    while True:
        try:
            mes_ini = int(input("Mês inicial (1-12): ").strip())
            ano_ini = int(input("Ano inicial (ex: 2000): ").strip())
            mes_fim = int(input("Mês final (1-12): ").strip())
            ano_fim = int(input("Ano final (ex: 2010): ").strip())
        except Exception:
            print("Entrada inválida. Digite números inteiros. Tente novamente.\n")
            continue

        if not (1 <= mes_ini <= 12 and 1 <= mes_fim <= 12):
            print("Mês inválido. Deve ser entre 1 e 12.\n")
            continue
        #Anos no intervalo do arquivo
        if ano_ini < ano_min or ano_fim > ano_max:
            print(f"Anos fora do intervalo dos dados ({ano_min} a {ano_max}). Ajuste.\n")
            continue
        #Período válido
        if (ano_ini > ano_fim) or (ano_ini == ano_fim and mes_ini > mes_fim):
            print("Data inicial posterior à final. Tente novamente.\n")
            continue
        break

    #Escolha da opção
    while True:
        print("\nEscolha a visualização:")
        print("1 - Todos os dados")
        print("2 - Apenas precipitação")
        print("3 - Apenas temperatura (máx e mín)")
        print("4 - Apenas umidade e vento")
        try:
            opcao = int(input("Opção (1-4): ").strip())
            if 1 <= opcao <= 4:
                break
            else:
                print("Opção inválida. Digite 1,2,3 ou 4.\n")
        except Exception:
            print("Entrada inválida. Digite um número.\n")

    #Cabeçalho conforme opção
    print("\n=== RESULTADO ===")
    if opcao == 1:
        print(f"{'DATA':10s} {'PRECIP(mm)':>10s} {'MAX(°C)':>9s} {'MIN(°C)':>9s} {'UMID(%)':>9s} {'VENTO(m/s)':>12s}")
    elif opcao == 2:
        print(f"{'DATA':10s} {'PRECIP(mm)':>10s}")
    elif opcao == 3:
        print(f"{'DATA':10s} {'MAX(°C)':>9s} {'MIN(°C)':>9s}")
    else:
        print(f"{'DATA':10s} {'UMID(%)':>9s} {'VENTO(m/s)':>12s}")

    #Percorre e imprime registros
    for r in dados:
        cond_inicio = (r["ano"] > ano_ini) or (r["ano"] == ano_ini and r["mes"] >= mes_ini)
        cond_fim = (r["ano"] < ano_fim) or (r["ano"] == ano_fim and r["mes"] <= mes_fim)
        if cond_inicio and cond_fim:
            if opcao == 1:
                print(f"{r['data']:10s} {r['precip']:10.1f} {r['maxima']:9.1f} {r['minima']:9.1f} {r['umidade']:9.1f} {r['vento']:12.1f}")
            elif opcao == 2:
                print(f"{r['data']:10s} {r['precip']:10.1f}")
            elif opcao == 3:
                print(f"{r['data']:10s} {r['maxima']:9.1f} {r['minima']:9.1f}")
            else:
                print(f"{r['data']:10s} {r['umidade']:9.1f} {r['vento']:12.1f}")

#B) Mês mais chuvoso
def mes_mais_chuvoso(dados):
    """Cria dicionário chave 'mm/aaaa' -> soma precip; retorna o mês/ano com maior soma."""
    if not dados:
        print("Nenhum dado carregado.")
        return
    precip_por_mes = {}
    for r in dados:
        chave = f"{r['mes']:02d}/{r['ano']}"
        if chave not in precip_por_mes:
            precip_por_mes[chave] = 0.0
        precip_por_mes[chave] += r['precip']

    #Maior precip
    chave_max = None
    valor_max = None
    for k, v in precip_por_mes.items():
        if chave_max is None or v > valor_max:
            chave_max = k
            valor_max = v

    #Exibir resultado
    mm, aaaa = chave_max.split("/")
    mm_i = int(mm)
    print("\n=== MÊS MAIS CHUVOSO ===")
    print(f"Mês/ano: {MESES_PT[mm_i]} {aaaa} ({chave_max})")
    print(f"Precipitação total no mês: {valor_max:.1f} mm")

    return precip_por_mes

#C)Média da temperatura mínima do mês
def medias_minimas_11anos(dados, mes):

    resultados = {}
    for ano in range(2006, 2017):
        soma = 0.0
        cont = 0
        for r in dados:
            if r["ano"] == ano and r["mes"] == mes:
                soma += r["minima"]
                cont += 1
        chave = f"{MESES_PT[mes]}{ano}"
        if cont > 0:
            resultados[chave] = soma / cont
        else:
            resultados[chave] = None
    return resultados

#D)Gráfico
def plot_min_temps(min_dict, mes):


    anos = []
    valores = []
    for chave in sorted(min_dict.keys(), key=lambda k: int(k[-4:])):
        anos.append(int(chave[-4:]))
        v = min_dict[chave]
        valores.append(v if v is not None else float('nan'))


    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(9,5))
        plt.bar(anos, valores)
        plt.xlabel("Ano")
        plt.ylabel("Média da temperatura mínima (°C)")
        plt.title(f"Média da temperatura mínima - {MESES_PT[mes]} (2006-2016)")
        plt.grid(axis='y', linestyle='--', linewidth=0.4)
        nome_arquivo = f"minimas_{mes}.png"
        plt.savefig(nome_arquivo, bbox_inches='tight')
        plt.show()
        print(f"Gráfico salvo como '{nome_arquivo}' na pasta atual.")
    except Exception:

        print("\n(matplotlib não disponível — mostrando gráfico em texto)")

        valores_validos = [v for v in valores if not (v != v)]
        if not valores_validos:
            print("Sem dados para plotar.")
            return
        maxv = max(valores_validos)
        for ano, v in zip(anos, valores):
            if v != v:  # NaN
                print(f"{ano}: sem dados")
            else:
                n_hash = int((v / maxv) * 40) if maxv != 0 else 0
                print(f"{ano}: {v:.2f} {'#' * n_hash}")

#E)Média geral das médias
def media_geral(min_dict):
    soma = 0.0
    cont = 0
    for v in min_dict.values():
        if v is not None:
            soma += v
            cont += 1
    if cont == 0:
        return None
    return soma / cont

#Ler mês do usuário
def ler_mes_usuario(prompt="Digite o mês (1-12): "):
    while True:
        try:
            mes = int(input(prompt).strip())
            if 1 <= mes <= 12:
                return mes
            else:
                print("Mês inválido. Informe 1..12.")
        except Exception:
            print("Entrada inválida. Digite um número inteiro.")

#Menu principal
def main():
    arquivo = "Anexo_Arquivo_Dados_Projeto_Logica_e_programacao_de_computadores.csv"
    print("Carregando dados (arquivo na mesma pasta):", arquivo)
    dados = carregar_dados(arquivo)
    if not dados:
        print("Erro: nenhum dado carregado. Verifique o arquivo e caminho.")
        return
    print(f"Dados carregados: {len(dados)} registros.\n")

    ultimo_dict_medias = None
    ultimo_mes_consultado = None

    while True:
        print("\n=== MENU ===")
        print("1 - Visualizar intervalo de dados (Item A)")
        print("2 - Mês mais chuvoso (Item B)")
        print("3 - Médias da temperatura mínima (Item C: 2006-2016)")
        print("4 - Gráfico das médias (Item D)")
        print("5 - Média geral das médias (Item E)")
        print("6 - Sair")
        opc = input("Escolha uma opção (1-6): ").strip()
        if opc == "1":
            visualizar_intervalo(dados)
        elif opc == "2":
            mes_mais_chuvoso(dados)
        elif opc == "3":
            mes = ler_mes_usuario("Informe o mês a analisar (1-12) para os anos 2006-2016: ")
            d = medias_minimas_11anos(dados, mes)
            print(f"\nMédias da temperatura mínima para {MESES_PT[mes]} (2006-2016):")
            for k, v in d.items():
                if v is None:
                    print(f"{k}: sem dados")
                else:
                    print(f"{k}: {v:.2f} °C")
            ultimo_dict_medias = d
            ultimo_mes_consultado = mes
        elif opc == "4":

            if ultimo_dict_medias is None:
                print("Ainda não calculou as médias (Item C). Vamos pedir o mês agora.")
                mes = ler_mes_usuario("Informe o mês a analisar (1-12) para os anos 2006-2016: ")
                ultimo_dict_medias = medias_minimas_11anos(dados, mes)
                ultimo_mes_consultado = mes
            plot_min_temps(ultimo_dict_medias, ultimo_mes_consultado)
        elif opc == "5":
            if ultimo_dict_medias is None:
                print("Ainda não calculou as médias (Item C). Vamos pedir o mês agora.")
                mes = ler_mes_usuario("Informe o mês a analisar (1-12) para os anos 2006-2016: ")
                ultimo_dict_medias = medias_minimas_11anos(dados, mes)
                ultimo_mes_consultado = mes
            mg = media_geral(ultimo_dict_medias)
            if mg is None:
                print("Não há dados suficientes para calcular a média geral.")
            else:
                print(f"Média geral da temperatura mínima em {MESES_PT[ultimo_mes_consultado]} (2006-2016): {mg:.2f} °C")
        elif opc == "6":
            print("Encerrando. Até mais!")
            break
        else:
            print("Opção inválida. Escolha 1..6.")

if __name__ == "__main__":
    main()


Carregando dados (arquivo na mesma pasta): Anexo_Arquivo_Dados_Projeto_Logica_e_programacao_de_computadores.csv
Arquivo não encontrado: Anexo_Arquivo_Dados_Projeto_Logica_e_programacao_de_computadores.csv
Erro: nenhum dado carregado. Verifique o arquivo e caminho.
